In [277]:
!pip install pandas_ta
!pip install yfinance

In [278]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

In [279]:
def get_dynamic_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch Wikipedia page. Status code: {response.status_code}")
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table', {'id': 'constituents'})
    ticker_list = []
    for row in table.find_all('tr')[1:]:
        columns = row.find_all('td')
        if columns:
            ticker = columns[0].text.strip()
            ticker_clean = ticker.replace('.', '-')
            ticker_list.append(ticker_clean)
    return ticker_list

In [280]:
sp500_tickers = get_dynamic_sp500_tickers()

In [281]:
df_bulk = yf.download(tickers=sp500_tickers, start="2015-01-01", end="2023-12-31", group_by="ticker")

/tmp/ipykernel_525/1360888205.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_bulk = yf.download(tickers=sp500_tickers, start="2015-01-01", end="2023-12-31", group_by="ticker")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[                       0%                       ]  2 of 503 completed/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[                       1

In [306]:
df_stacked = df_bulk.stack(level=0, future_stack=True).rename_axis(['Date', 'Ticker']).reset_index()
df_stacked = df_stacked.sort_values(by=['Date', 'Ticker'])
df_stacked.columns.name = None
df_stacked = df_stacked.set_index(['Date', 'Ticker'])

In [307]:
def calculate_features(group):
    # Group 1: pandas-ta features
    group.ta.rsi(length=14, append=True)
    group.ta.ppo(append=True)
    group.ta.obv(append=True)
    group.ta.atr(length=14, append=True)

    # Group 2: Custom Pandas Math
    group['Log_Return'] = np.log(group['Close'] / group['Close'].shift(1))
    group['Overnight_Gap'] = (group['Open'] - group['Close'].shift(1)) / group['Close'].shift(1)
    group['High_Low_Ratio'] = (group['High'] - group['Low']) / group['Close']

    return group

In [308]:
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
df_stacked[numeric_cols] = df_stacked[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [309]:
if 'Adj Close' in df_stacked.columns:
    df_stacked = df_stacked.drop(columns=['Adj Close'])

# 2. Protect Date and Ticker in the index
if 'Ticker' in df_stacked.columns:
    df_stacked = df_stacked.set_index(['Date', 'Ticker'])
df_stacked = df_stacked.dropna()

In [310]:
df_stacked.head()

Open       High        Low      Close       Volume
Date       Ticker                                                         
2015-01-02 A       37.621143  37.739909  36.881144  37.054726    1529200.0
           AAPL    24.671149  24.682224  23.776352  24.214891  212818400.0
           ABBV    41.470312  42.078678  41.470312  41.755482    5086100.0
           ABT     36.517574  36.678978  36.025292  36.235119    3216600.0
           ACGL    18.764398  18.884845  18.472788  18.539352    1101600.0

In [311]:
df_features = df_stacked.groupby(level='Ticker', group_keys=False).apply(calculate_features)
df_features = df_features.dropna().reset_index()

In [312]:
df_features.head()

,Date,Ticker,Open,High,Low,Close,Volume,RSI_14,PPO_12_26_9,PPOh_12_26_9,PPOs_12_26_9,OBV,ATRr_14,Log_Return,Overnight_Gap,High_Low_Ratio
0,2015-02-09,A,35.839674,36.022391,35.583874,35.666096,3586100.0,38.808654,-0.732854,0.0,-0.732854,-1.332470e+07,0.797913,-0.007655,-0.002796,0.012295
1,2015-02-09,AAPL,26.360603,26.647444,26.333919,26.620762,155559200.0,49.439740,3.792268,0.0,3.792268,2.051296e+09,0.639402,0.006621,-0.003195,0.011777
2,2015-02-09,ABBV,35.543756,35.977917,35.230906,35.422447,22522000.0,24.094962,-3.528696,0.0,-3.528696,-5.746540e+07,1.328799,-0.025272,-0.021617,0.021089
3,2015-02-09,ABT,36.678438,36.678438,35.940164,36.118649,6585900.0,48.284519,0.228643,0.0,0.228643,-1.972720e+07,0.732736,-0.021773,-0.006373,0.020440
4,2015-02-09,ACGL,19.024309,19.078193,18.926050,18.945066,1284300.0,48.325565,0.509048,0.0,0.509048,3.255900e+06,0.296094,-0.004007,0.000167,0.008031


In [313]:
# Restore the correct 5 macroeconomic benchmarks
benchmark_tickers = ['^TNX', '^IRX', '^VIX', 'DX-Y.NYB', 'XLK']

print("Downloading Macroeconomic Benchmarks...")
df_bench = yf.download(benchmark_tickers, start="2015-01-01", end="2023-12-31")['Close']

# Calculate Macro Features
df_macro = pd.DataFrame(index=df_bench.index)
df_macro['Term_Spread'] = df_bench['^TNX'] - df_bench['^IRX']
df_macro['VIX'] = df_bench['^VIX']
df_macro['USD_Index'] = df_bench['DX-Y.NYB']
df_macro['XLK_Log_Return'] = np.log(df_bench['XLK'] / df_bench['XLK'].shift(1))
df_macro = df_macro.dropna().reset_index()

# ==========================================
# 1. THE TUPLE CRUSHER (Flatten all columns)
# ==========================================
df_features.columns = [col[0] if isinstance(col, tuple) else col for col in df_features.columns]
df_macro.columns = [col[0] if isinstance(col, tuple) else col for col in df_macro.columns]

if isinstance(df_features.columns, pd.MultiIndex):
    df_features.columns = df_features.columns.get_level_values(0)

# ==========================================
# 2. THE DATE COLUMN FIX
# ==========================================
# Fix df_macro's Date Column
if 'Date' not in df_macro.columns:
    df_macro = df_macro.rename(columns={df_macro.columns[0]: 'Date'})

# Fix df_features' Date Column
if 'Ticker' not in df_features.columns:
    df_features = df_features.reset_index()
if 'Date' not in df_features.columns:
    df_features = df_features.rename(columns={df_features.columns[0]: 'Date'})

# Standardize Datetimes
df_features['Date'] = pd.to_datetime(df_features['Date']).dt.tz_localize(None)
df_macro['Date'] = pd.to_datetime(df_macro['Date']).dt.tz_localize(None)

# ==========================================
# 3. MERGING
# ==========================================
print("Merging Macro features into the main dataset...")
df_final = pd.merge(df_features, df_macro, on='Date', how='left')

df_final['Sector_Relative_Strength'] = df_final['Log_Return'] - df_final['XLK_Log_Return']
df_final = df_final.drop(columns=['XLK_Log_Return'])
df_final = df_final.sort_values(by=['Date', 'Ticker'])

print("\nSuccessfully Merged Data:")
print(df_final.head(5))

/tmp/ipykernel_525/835735743.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_bench = yf.download(benchmark_tickers, start="2015-01-01", end="2023-12-31")['Close']
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[                       0%                       ]

/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*******************   40%                       ]  2 of 5 completed/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  5 of 5 completed


Merging Macro features into the main dataset...

Successfully Merged Data:
        Date Ticker       Open       High        Low      Close       Volume  \
0 2015-02-09      A  35.839674  36.022391  35.583874  35.666096    3586100.0   
1 2015-02-09   AAPL  26.360603  26.647444  26.333919  26.620762  155559200.0   
2 2015-02-09   ABBV  35.543756  35.977917  35.230906  35.422447   22522000.0   
3 2015-02-09    ABT  36.678438  36.678438  35.940164  36.118649    6585900.0   
4 2015-02-09   ACGL  19.024309  19.078193  18.926050  18.945066    1284300.0   

      RSI_14  PPO_12_26_9  PPOh_12_26_9  PPOs_12_26_9           OBV   ATRr_14  \
0  38.808654    -0.732854           0.0     -0.732854 -1.332470e+07  0.797913   
1  49.439740     3.792268           0.0      3.792268  2.051296e+09  0.639402   
2  24.094962    -3.528696           0.0     -3.528696 -5.746540e+07  1.328799   
3  48.284519     0.228643           0.0      0.228643 -1.972720e+07  0.732736   
4  48.325565     0.509048           0.0

# First Model

In [314]:
# ==========================================
# 0. DEFINE THE FEATURES
# ==========================================
features = [
    'RSI_14', 'PPO_12_26_9', 'PPOh_12_26_9', 'PPOs_12_26_9', 'OBV', 'ATRr_14',
    'Log_Return', 'Overnight_Gap', 'High_Low_Ratio', 'Term_Spread', 'VIX',
    'USD_Index', 'Sector_Relative_Strength'
]

# ==========================================
# 1. CREATE TARGETS ON THE MAIN DATASET
# ==========================================
# Make sure the data is sorted chronologically before shifting!
df_final = df_final.sort_values(by=['Date', 'Ticker'])

# Calculate the actual return for evaluating your strategy's profit
df_final['Next_Close'] = df_final.groupby('Ticker')['Close'].shift(-1)
df_final['Next_Day_Return'] = (df_final['Next_Close'] - df_final['Close']) / df_final['Close']
df_final = df_final.dropna()

# Calculate the 0-4 Integer Score for the AI to train on
df_final['Relevance_Score'] = df_final.groupby('Date')['Next_Day_Return'].transform(
    lambda x: pd.qcut(x, q=5, labels=False, duplicates='drop')
)

# ==========================================
# 2. PERFORM THE TIME-SERIES SPLIT
# ==========================================
split_date = '2022-01-01'
train_data = df_final[df_final['Date'] < split_date]
test_data  = df_final[df_final['Date'] >= split_date]

# We must drop any remaining NaNs to prevent XGBoost from crashing
train_data = train_data.dropna(subset=['Relevance_Score'])

# ------------------------------------------
# CRITICAL FIX: The AI trains on Relevance_Score,
# but we keep Next_Day_Return for the profit calculation!
# ------------------------------------------
X_train = train_data[features]
y_train = train_data['Relevance_Score']

X_test = test_data[features]
y_test = test_data['Next_Day_Return']

# ==========================================
# 3. CREATE GROUP ARRAYS & TRAIN
# ==========================================
train_groups = train_data.groupby('Date').size().values
test_groups = test_data.groupby('Date').size().values

ranker = xgb.XGBRanker(
    objective='rank:pairwise',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.6,
    colsample_bytree=0.6,
    random_state=42
)

print("Training XGBRanker on Integer Relevance Scores (0 to 4)...")
ranker.fit(X_train, y_train, group=train_groups)

test_data_eval = test_data.copy()
test_data_eval['Predicted_Score'] = ranker.predict(X_test)
print("Training & Predictions Complete!")

Training XGBRanker on Integer Relevance Scores (0 to 4)...
Training & Predictions Complete!


In [315]:
daily_results = [] # Always reset the list!

# Loop through and buy the #1 AI pick each day
for date, group in test_data_eval.groupby('Date'):
    sorted_day = group.sort_values(by='Predicted_Score', ascending=False)
    top_stock = sorted_day.iloc[0]
    daily_results.append(top_stock['Next_Day_Return'])

daily_array = np.array(daily_results)

# TRUE Compounding Math
strategy_total_return = (np.prod(1 + daily_array) - 1) * 100
average_daily_return = np.mean(daily_array) * 100

print(f"Total Strategy Return (Testing Period): {strategy_total_return:.2f}%")
print(f"Average Daily Return of Top Pick:       {average_daily_return:.3f}%")

Total Strategy Return (Testing Period): -10.43%
Average Daily Return of Top Pick:       0.206%


In [316]:
test_date = '2023-11-09'
sample_day = test_data_eval[test_data_eval['Date'] == test_date]
sample_day_sorted = sample_day.sort_values(by='Predicted_Score', ascending=False)
print("\nAI Rankings for ", test_date, ":")
print(sample_day_sorted[['Ticker', 'Predicted_Score', 'Next_Day_Return']])


AI Rankings for  2023-11-09 :
        Ticker  Predicted_Score  Next_Day_Return
1063883   ALGN         0.262081         0.025569
1064217   PAYC         0.191126         0.020267
1064177   MRNA         0.131678         0.007769
1064251     RF         0.111426         0.006658
1064185    MTD         0.098418         0.019091
...        ...              ...              ...
1064054    FIX        -0.243039         0.013063
1064336    WAB        -0.248675         0.010507
1064104    IFF        -0.257820         0.002796
1064226    PGR        -0.275600         0.000250
1063937    CAH        -0.289198         0.012742

[499 rows x 3 columns]
